<a href="https://colab.research.google.com/github/MathewBiddle/map-of-activities/blob/main/MBON_harvest_registration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install selenium beautifulsoup4 requests

In [2]:
## For google spreadsheet reading you need to authenticate w/ google

import pandas as pd

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [3]:
def get_sheet_data(url: str, sheet: str) -> pd.DataFrame:
  '''
  Gets data from specific worksheet in Google Spreadsheet and
  returns as a DataFrame.

  parameters:
  sheet (str): name of worksheet, eg 'Form Responses 1'

  returns:
  DataFrame: worksheet data as a DataFrame

  '''

  worksheet = gc.open_by_url(url)
  room_n = worksheet.worksheet(sheet)

  df = pd.DataFrame(room_n.get_all_records())

  return df

In [4]:
url = 'https://docs.google.com/spreadsheets/d/1jBS8ASS27yV8APZ8Fh-tgX6dHdopwianrUZv0kbKcxw/edit?gid=1698140136#gid=1698140136'
sheet = 'Form Responses 1'

df = get_sheet_data(url, sheet)

df = df.replace('(?i)yes$',True, regex=True).replace('(?i)no$',False,regex=True)

df.sample(4)

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,Upload the dataset.,Are there scripts or code used to process the data?,"If yes to above, and they are publicly available, please include appropriate link(s) here.","If yes to above, and the code is available, please include appropriate link(s) here.",Would you like this dataset visualized in the MBON Data Portal (https://mbon.ioos.us/)?,Email Address,"If the dataset is already visualized in the MBON data portal, please include the link(s) to the data layer(s) here.",,What is the expected timeline for this dataset?,additonal comments
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,,,,,
22,12/16/2021 22:55:06,NERACOOS ISMN-MBON Gulf of Maine Plankton Obse...,Wilkinson Basin Time Series Station (WBTS),,Dylan Pugh,dpugh@gmri.org,Jeffrey Runge,jeffrey.runge@maine.edu,Gulf of Maine,NERACOOS,...,,True,,,True,dpugh@gmri.org,,,,
33,10/19/2021 14:22,SBC LTER: Beach: Time-series of beach wrack co...,"composition, count, and wet biomass of macroin...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,,,,,
47,9/30/2021 18:42:53,18S Monterey Bay Time Series: an eDNA data set...,These data are from marine filtered seawater s...,,"Marine Lebrec, Kathleen Pitz","mlebrec@mbari.org, kpitz@mbari.org","Francisco Chavez, Kathleen Pitz","chfr@mbari.org, kpitz@mbari.org",Central California,CeNCOOS,...,,True,https://github.com/iobis/dataset-edna,,True,mlebrec@mbari.org; kpitz@mbari.org,,,"Ideally, this would be done by the end of the ...",


## Function to extract json-ld from webpage

In [32]:
from selenium import webdriver
from bs4 import BeautifulSoup
import json
import requests

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')  # Run Chrome in headless mode
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

def get_ld_json(url: str) -> dict:
    parser = "lxml"

    browser = webdriver.Chrome(options=chrome_options)
    browser.get(url)
    html_source = browser.page_source
    soup = BeautifulSoup(html_source, parser)

    # faster but sometimes doesn't work
    #req = requests.get(url)
    #soup = BeautifulSoup(req.text, parser)

    return json.loads("".join(soup.find("script", {"type":"application/ld+json"}).contents))

## Check through datasets that have a link to a repository and extract json-ld

In [33]:
df['spatialCoverage'] = pd.Series()

for index, row in df.loc[df['If yes to above, please include appropriate link(s) here.']!='',['If yes to above, please include appropriate link(s) here.','Dataset title']].iterrows():

  url = row['If yes to above, please include appropriate link(s) here.']
  title = row['Dataset title']


  if url.startswith('10.154'):
    url = f'https://dx.doi.org/{url}'
    #print(f'{url}\n')

  elif '\ndata can also be accessed by using the repository\'s API.' in url:
    url = url.replace('data can also be accessed by using the repository\'s API.','')
    #print(f'{url}\n')

  elif url.endswith('.pdf'):
    continue

  elif 'usf.box.com' in url:
    continue

  elif url == 'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=91':
    url = 'https://portal.edirepository.org/nis/mapbrowse?scope=knb-lter-sbc&identifier=91'

  elif url == 'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=51':
    url = 'https://portal.edirepository.org/nis/mapbrowse?scope=knb-lter-sbc&identifier=51'
    # has a list of GeoCoordinates

  # elif 'neracoos.org/erddap' in url:
  #   # NERACOOS is running an old ERDDAP which represents
  #   # schema.org bounding box incorrectly. See
  #   # https://erddap.github.io/changes#version-200
  #   continue


  try:
    get_ld_json(url)

    if 'ncei' in url:
      spatial = get_ld_json(url)['spatialCoverage'][0]['geo']
    else:
      spatial = get_ld_json(url)['spatialCoverage']['geo']

    print(f'{url} has spatial {spatial}')

    if len(spatial) > 1:
      # just take the first spatial coordinates
      df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial[0]]
    elif len(spatial) == 1:
      df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial]

  except:
    print(f'{url} cant find json-ld')

  # if len(spatial) > 1:
  #     # just take the first one
  #     df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial[0]]
  # elif len(spatial) == 1:
  #     df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial]

https://dx.doi.org/10.15468/bfd6ci cant find json-ld
https://dx.doi.org/10.15468/h585qq has spatial {'@type': 'GeoShape', 'box': '24.647 26.785 -82.705 -79.277'}
https://dx.doi.org/10.15468/h585qq cant find json-ld
https://grunt.sefsc.noaa.gov/rvc_analysis20/samples/index cant find json-ld
https://ecotaxa.obs-vlfr.fr/prj/9989 cant find json-ld
https://data.piscoweb.org/metacatui/view/doi%3A10.6085%2FAA%2Fmarine_cbs.5.6 cant find json-ld
https://pubmed.ncbi.nlm.nih.gov/29937700/ cant find json-ld
https://dx.doi.org/10.15468/buqg4u  has spatial {'@type': 'GeoShape', 'box': '24.476 25.006 -81.715 -80.379'}
https://dx.doi.org/10.15468/buqg4u  cant find json-ld
https://data.neracoos.org/erddap/info/WBTS_CFIN_2004_2017/index.html has spatial {'@type': 'GeoShape', 'box': '42.85 -69.8822 42.9593 -69.53'}
https://data.neracoos.org/erddap/info/WBTS_CFIN_2004_2017/index.html cant find json-ld
https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=5 has spatial [{'@type': 'GeoShape', 

In [25]:
spatial

[{'@type': 'GeoShape', 'box': '32.8 -120.6344833 34.87315 -118.4'}]

In [30]:
# title = 'SBC LTER: Beach: Time-series of beach wrack consumers, ongoing since 2011'

# url = df.loc[df['Dataset title']==title,'If yes to above, please include appropriate link(s) here.'][33]

# if url == 'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=91':
#   print(f'Changing {url} to')
#   url = 'https://portal.edirepository.org/nis/mapbrowse?scope=knb-lter-sbc&identifier=91'
#   print(f'{url}')

url = 'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=5'

spatial = get_ld_json(url)#['spatialCoverage']['geo']

spatial

# df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial[0]]

# df.loc[df['Dataset title']==title].T
#df.loc[df['Dataset title']=='SBC LTER: Beach: Time-series of beach wrack consumers, ongoing since 2011','If yes to above, please include appropriate link(s) here.'][33]

{'@context': {'@vocab': 'https://schema.org/',
  'prov': 'http://www.w3.org/ns/prov#',
  'spdx': 'http://spdx.org/rdf/terms#'},
 '@type': 'Dataset',
 'name': 'Santa Barbara Channel Marine BON: Nearshore kelp forest integrated fish, 1981-ongoing',
 'description': 'The Santa Barbara Channel Marine Biodiversity Observation Network\n    (SBCMBON) tracks long-term patterns in species abundance and\n    diversity. This dataset contains counts of fish (including cryptic\n    fish, which are deliberately sought out) produced by integrating\n    data from four contributing projects working in the kelp forests of\n    the Santa Barbara Channel, USA.\n  \n      \n    The four contributing projects are two research projects, the Santa\n    Barbara Coastal LTER (SBC LTER) and the Partnership for\n    Interdisciplinary Studies of Coastal Oceans (PISCO), and the kelp\n    forest monitoring program of the Santa Barbara Channel National\n    Park, and the San Nicolas Island monitoring program supported

In [31]:
spatial['spatialCoverage']['geo']

[{'@type': 'GeoShape', 'box': '32.8 -120.6344833 34.87315 -118.4'}]

In [12]:
df.loc[df['If yes to above, please include appropriate link(s) here.']!='',['If yes to above, please include appropriate link(s) here.','Dataset title']]

,"If yes to above, please include appropriate link(s) here.",Dataset title
0,10.15468/bfd6ci,Marine Invertebrate Voucher Specimens (FWC-Col...
2,10.15468/h585qq,South Florida Fisheries Habitat Assessment (FW...
6,https://safmc.net/wp-content/uploads/2022/05/S...,South Atlantic Ecopath with Ecosim Ecosystem M...
9,https://usf.box.com/s/dvoi1ve0jn3apbdlad114uhn...,FKNMS Cruises CTD data
11,https://grunt.sefsc.noaa.gov/rvc_analysis20/sa...,Florida Keys and Dry Tortugas Reef Visual Cens...
12,https://ecotaxa.obs-vlfr.fr/prj/9989,"Time series of image-based, in vivo plankton a..."
18,https://data.piscoweb.org/metacatui/view/doi%3...,MARINe/PISCO: Intertidal: MARINe Coastal Biodi...
20,https://pubmed.ncbi.nlm.nih.gov/29937700/,Marine zooplankton community from coral reef s...
21,10.15468/buqg4u,Time series of zooplankton abundance in South ...
22,https://data.neracoos.org/erddap/info/WBTS_CFI...,NERACOOS ISMN-MBON Gulf of Maine Plankton Obse...


## Extract coordinate information

Add coordinate information from schema.org back into the DataFrame.

In [13]:
df = pd.concat([df, pd.json_normalize(df['spatialCoverage'])], axis=1)

df.loc[~df['spatialCoverage'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,Email Address,"If the dataset is already visualized in the MBON data portal, please include the link(s) to the data layer(s) here.",,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN
28,10/19/2021 14:22,SBC MBON: Benthic percent cover in the Santa B...,This data set contains percent cover and occur...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.01...",GeoCoordinates,NaN,34.01683333,-119.361475
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '31.9 -121.15 34....",GeoShape,31.9 -121.15 34.5 -117.27,NaN,NaN
32,10/19/2021 14:22,Plumes and Blooms: Curated oceanographic and p...,"The Plumes and Blooms project (PnB), has condu...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,"{'@type': 'GeoShape', 'box': '33.3 -79.21 33.3...",GeoShape,33.3 -79.21 33.38 -79.17,NaN,NaN
33,10/19/2021 14:22,SBC LTER: Beach: Time-series of beach wrack co...,"composition, count, and wet biomass of macroin...",,"Information Manager, Southern California Bight

## Box

In [14]:
import geopandas as gpd

test = df.loc[~df['box'].isna()]

# read from 'box'
def box_to_wkt(box_str):
       """Converts a box string to WKT format."""
       try:
           # Assuming box_str is in the format 'north, west, south, east'
           south, west, north, east = map(float, box_str.split(' '))
           # Create WKT polygon string
           wkt_polygon = f'POLYGON(({west} {north}, {east} {north}, {east} {south}, {west} {south}, {west} {north}))'
           return wkt_polygon
       except (ValueError, AttributeError):
           # Handle cases where box_str is not in the expected format or is None
           return None

test['wkt'] = test['box'].apply(box_to_wkt)

#test['geometry'] = gpd.GeoSeries.from_wkt(test['wkt'], crs='EPSG:4326')

#test

df.loc[~df['box'].isna(),'wkt'] = test['wkt']

df.loc[~df['box'].isna()]
# read 'box' into geometry somehow
#gpd.GeoSeries.from_wkt(df['box'])

<ipython-input-14-65551a2ccba0>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['wkt'] = test['box'].apply(box_to_wkt)


,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,"If the dataset is already visualized in the MBON data portal, please include the link(s) to the data layer(s) here.",,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,wkt
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ..."
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN,"POLYGON((-119.52 34.01, -119.45 34.01, -119.45..."
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '31.9 -121.15 34....",GeoShape,31.9 -121.15 34.5 -117.27,NaN,NaN,"POLYGON((-121.15 34.5, -117.27 34.5, -117.27 3..."
32,10/19/2021 14:22,Plumes and Blooms: Curated oceanographic and p...,"The Plumes and Blooms project (PnB), has condu...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '33.3 -79.21 33.3...",GeoShape,33.3 -79.21 33.38 -79.17,NaN,NaN,"POLYGON((-79.21 33.38, -79.17 33.38, -79.17 33..."
34,10/19/2021 14:22,SBC LTER: Beach: Time series of abundance of b...,"distribution, abundance and seasonal occurrenc...",,"Infor

In [15]:
df.loc[~df['box'].isna(),'box']

,box
23,32.8 -120.6344833 34.87315 -118.4
24,32.8 -120.65022 34.87315 -118.4
25,32.8 -120.65022 34.87315 -118.4
26,27.01 -124.77 48.40 -114.04
27,34.4030 -120.0669 34.4611 -119.8660
29,33.91 -119.52 34.01 -119.45
31,31.9 -121.15 34.5 -117.27
32,33.3 -79.21 33.38 -79.17
34,33.185566666666666 -81.75708055555556 33.37228...
36,33.56 -120.73 34.61 -118.11


## Polygon

In [17]:
df.loc[~df['polygon'].isna()]

KeyError: 'polygon'

In [16]:
temp=df.loc[~df['polygon'].isna()]

temp['wkt'] = 'POLYGON ((' + temp['polygon'].astype(str) + '))'

df.loc[~df['polygon'].isna(),'wkt'] = temp['wkt']

# gs = gpd.GeoSeries.from_wkt(temp['wkt'])

# gdf2 = gpd.GeoDataFrame(
#     temp,
#     geometry=gs,
#     crs='EPSG:4326'
#     )

# gdf.loc[~gdf['polygon'].isna(),'geometry'] = gdf2['geometry']


df.loc[~df['polygon'].isna()]
#gdf.merge(gdf2, how='inner', on='Dataset title')

KeyError: 'polygon'

## GeoCoordinates

In [34]:
import geopandas as gpd
temp = df.loc[df['@type']=='GeoCoordinates', ['latitude','longitude']]


gdf = gpd.GeoDataFrame(
    temp, geometry=gpd.points_from_xy(
        temp['longitude'],
        temp['latitude'])
    )

df.loc[df['@type']=='GeoCoordinates','geometry'] = gdf['geometry']

df.loc[df['@type']=='GeoCoordinates']

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,wkt,geometry
28,10/19/2021 14:22,SBC MBON: Benthic percent cover in the Santa B...,This data set contains percent cover and occur...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.01...",GeoCoordinates,NaN,34.01683333,-119.361475,NaN,POINT (-119.36148 34.01683)
33,10/19/2021 14:22,SBC LTER: Beach: Time-series of beach wrack co...,"composition, count, and wet biomass of macroin...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.40...",GeoCoordinates,NaN,34.40305,-119.74375,NaN,POINT (-119.74375 34.40305)
35,10/19/2021 14:22,Abundance and species composition of benthic h...,counts of heterobranch molluscs (sea slugs and...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.43...",GeoCoordinates,NaN,34.4339,-119.95,NaN,POINT (-119.95 34.4339)
39,10/19/2021 14:22,Data to support manuscript: A Comparison of Tw...,fish dataset was collected at platform Harmony...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.37...",GeoCoordinates,NaN,34.37,-120.16,NaN,POINT (-120.16 34.37)
40,10/19/2021 14:22,SBC LTER: Reef: Sightings of sea otters (Enhyd...,"number, location and activity or behavior of s...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.40...",GeoCoordinates,NaN,34.400275,-119.7445915,NaN,POINT (-119.74459 34.40028)
41,10/19/2021 14:22,Santa Barbara Channel Marine BON: Gray Whales ...,dataset documents the passage of gray whales (...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.40...",GeoCoordinates,NaN,34.40766,-119.877969,NaN,POINT (-119.87797 34.40766)
44,10/19/2021 14:22,SBC LTER and MBON: Sea urchin microbiomes in S...,data describe a comparative study of the micro...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': ' 34.4...",GeoCoordinates,NaN,34.416667,-119.95,NaN,POINT (-119.95 34.41667)


In [35]:
# wkt to geometry

temp = df.loc[~df['wkt'].isna()]

gs = gpd.GeoSeries.from_wkt(temp['wkt'])

gdf2 = gpd.GeoDataFrame(
    temp,
    geometry=gs,
    crs='EPSG:4326'
    )

df.loc[~df['wkt'].isna(),'geometry'] = gdf2['geometry']

df.loc[~df['wkt'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,wkt,geometry
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731...","POLYGON ((-120.63448 34.87315, -118.4 34.87315..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2...","POLYGON ((-124.77 48.4, -114.04 48.4, -114.04 ..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ...","POLYGON ((-120.0669 34.4611, -119.866 34.4611,..."
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN,"POLYGON((-119.52 34.01, -119.45 34.01, -119.45...","POLYGON ((-119.52 34.01, -119.45 34.01, -119.4..."
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '31.9 -121.15 34....",GeoShape,31.9 -121.15 34.5 -117.27,NaN,NaN,"POLYGON((-121.15 34.5, -117.27 34.5, -117.27 3...","POLYGON ((-121.15 34.5, -117.27 34.5, -117.27 ..."
32,10/19/2021 14:22,Plumes and Blooms: Curated oceanographic and p...,"The Plumes and Blooms project (PnB), has condu...",,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box'

In [36]:
df.loc[~df['@type'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,wkt,geometry
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731...","POLYGON ((-120.63448 34.87315, -118.4 34.87315..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2...","POLYGON ((-124.77 48.4, -114.04 48.4, -114.04 ..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ...","POLYGON ((-120.0669 34.4611, -119.866 34.4611,..."
28,10/19/2021 14:22,SBC MBON: Benthic percent cover in the Santa B...,This data set contains percent cover and occur...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoCoordinates', 'latitude': '34.01...",GeoCoordinates,NaN,34.01683333,-119.361475,NaN,POINT (-119.36148 34.01683)
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN,"POLYGON((-119.52 34.01, -119.45 34.01, -119.45...","POLYGON ((-119.52 34.01, -119.45 34.01, -119.4..."
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '31.9 -121.15 34....",GeoShape,31.9 -121.15 34.5 -117.27,NaN,NaN,"POLYGO

In [37]:
!pip install folium matplotlib mapclassify

In [38]:
gdf = gpd.GeoDataFrame(df, geometry=df['geometry'], crs = 'epsg:4326')

gdf

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,wkt,geometry
0,9/10/2024 15:30:54,Marine Invertebrate Voucher Specimens (FWC-Col...,taxa occurrence. 1950-present,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
1,9/10/2024 15:29:13,SEMAP-South Atlantic trawl surveys,1983-Present. conversion to DarwinCore in prog...,,"Jennifer Dorton, Kyle Wilcox","jdorton@secoora.org, Kyle@axiomdatascience.com","Jennifer Dorton, Kyle Wilcox","jdorton@secoora.org, Kyle@axiomdatascience.com",SE US,,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
3,9/10/2024 15:25:10,FKNMS WS Pigment Phytoplankton,generation of taxa output in progress.,,Sebastian,sebastian15@usf.edu,Sebastian,sebastian15@usf.edu,SE US,,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
4,9/10/2024 15:23:24,Walton Smith Primary Productivity,On Ian Smith's desktop,,Ian Smith,Ian.Smith@noaa.gov,Ian Smith,Ian.Smith@noaa.gov,SE US,,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,,"Marine Bird Sighting Data, Arctic Marine Biodi...",,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
60,,Vessel line-transect surveys of Arctic cetacea...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
61,,Vessel line-transect surveys of Arctic pinnipe...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None
62,,Vessel line-transect surveys of Arctic marine ...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,None


In [39]:
gdf.columns.tolist()

['Timestamp',
 'Dataset title',
 'Dataset summary',
 'Crossfunded? (Yes/No)',
 'Who is the data management POC for the dataset? ',
 'Data management POC email:',
 'Who is the technical POC for the dataset? ',
 'Technical POC email:',
 'Which MBON project is this dataset associated with?',
 'If you have worked with a Regional Association, please indicate which one(s).',
 'Are there any deadlines associated with this dataset?',
 'If one exists, enter the DOI for the dataset.',
 'If one exists, enter a citation for the dataset.',
 'Are the data accessible via the web?',
 'If yes to above, please include appropriate link(s) here.',
 'Has the dataset been loaded into ERDDAP?',
 'If yes to above, please include appropriate ERDDAP link(s) here.',
 'Has the dataset been translated into DarwinCore?',
 'Has the dataset been submitted to OBIS?',
 'If yes to above, please include appropriate OBIS link(s) here.',
 'Has the dataset been archived at NCEI?',
 'If yes to above, please include appropria

In [40]:
gdf[(gdf['Dataset title']!= 'South Florida Fisheries Habitat Assessment (FWC-Seagrass)') & (gdf['Dataset title']!= 'Time series of zooplankton abundance in South Florida from 2015 onward (MBON program)')].explore(tooltip=['Dataset title','spatialCoverage'], popup=['Dataset title','spatialCoverage','wkt','If yes to above, please include appropriate link(s) here.'])

In [41]:
gdf.loc[gdf['Dataset title'] == 'South Florida Fisheries Habitat Assessment (FWC-Seagrass)',['box','wkt','If yes to above, please include appropriate link(s) here.']]

,box,wkt,"If yes to above, please include appropriate link(s) here."
2,NaN,NaN,10.15468/h585qq


In [42]:
gdf.loc[gdf['Dataset title'] == 'Time series of zooplankton abundance in South Florida from 2015 onward (MBON program)', ['box','wkt','If yes to above, please include appropriate link(s) here.']]

,box,wkt,"If yes to above, please include appropriate link(s) here."
21,NaN,NaN,10.15468/buqg4u


In [ ]:
gdf.loc[gdf['Dataset title'] == 'SBC LTER: Time series of quarterly NetCDF files of kelp biomass in the canopy from Landsat 5, 7 and 8, since 1984 (ongoing)', ['box','wkt','If yes to above, please include appropriate link(s) here.']]

In [ ]:
df['If yes to above, please include appropriate link(s) here.'].unique()

In [ ]:
# script =<script src="https://code.jquery.com/jquery-3.6.0.slim.min.js" integrity="sha256-u7e5khyithlIdTpu22PHhENmPcRdFiHRjhAuHcs05RI=" crossorigin="anonymous"></script>
#         <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.min.js"></script>
#         <script>
#         $(document).ready( function () {
#             $('#table').DataTable();
#         } );
#         let table = new DataTable('#table');
#         </script>

In [ ]:
gdf[(gdf['Dataset title']== 'South Florida Fisheries Habitat Assessment (FWC-Seagrass)') | (gdf['Dataset title']== 'Time series of zooplankton abundance in South Florida from 2015 onward (MBON program)')].explore()

In [ ]:
gdf.loc[~gdf['@type'].isna()]